<a href="https://colab.research.google.com/github/ValentinaEmili/Texture-synthesis/blob/main/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install lpips

In [ ]:
REPO_DIR = '/content/Texture-synthesis'
os.chdir('/content')

if not os.path.exists(REPO_DIR):
  !git clone https://github.com/ValentinaEmili/Texture-synthesis.git

if REPO_DIR not in sys.path:
  sys.path.append(REPO_DIR)

In [ ]:
import os
from PIL import Image
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.optim as optim
from torchvision.models import vgg16
import lpips
import matplotlib.pyplot as plt
import time
import numpy as np
import math
import sys
from Texture_synthesis.VQ_VAE import VQ_VAE

In [ ]:
train_transform = transforms.Compose([
        transforms.Resize(512),
        transforms.RandomCrop(512),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

eval_transform = transforms.Compose([
        transforms.Resize(512),
        transforms.CenterCrop(512),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])


class DTD_Dataset(Dataset):
    def __init__(self, root, file_list, transform=None, class_to_idx=None):
        self.root = root
        self.transform = transform

        with open(file_list, mode='r', encoding='utf-8') as f:
            self.files = [line.strip() for line in f if line.strip()]

        if class_to_idx is None:
          unique_classes = sorted({os.path.normpath(p).split(os.sep)[0] for p in self.files})
          self.class_to_idx = {class_name: i for i, class_name in enumerate(unique_classes)}
        else:
          self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        relative_path = self.files[idx]
        image_path = os.path.join(self.root, relative_path)
        img = Image.open(image_path).convert('RGB')
        rel_norm = os.path.normpath(relative_path)
        string_label = rel_norm.split(os.sep, 1)[0]
        label = self.class_to_idx[string_label]

        if self.transform:
            img = self.transform(img)

        return img, label

In [ ]:
path_images = "drive/MyDrive/DeepLearning/dtd/images"
path_labels = "drive/MyDrive/DeepLearning/dtd/labels"
train_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "train1.txt"), train_transform)
class_to_idx = train_dataset.class_to_idx
val_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "val1.txt"), eval_transform, class_to_idx=class_to_idx)
test_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "test1.txt"), eval_transform, class_to_idx=class_to_idx)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VQ_VAE().to(device)
optimizer = optim.Adam(model.parameters(), lr=5e-5, betas=(0.9, 0.999)) # for VQ-VAE
perceptual_loss_fn = lpips.LPIPS(net='vgg').to(device)

In [ ]:
def visual_validation(model, val_loader, device):
  model.eval()

  all_indices = []
  visual_samples = None
  total_percept, total_recon, total_vq = 0.0, 0.0, 0.0

  num_embeddings = model.vq.num_embeddings
  embedding_dim = model.vq.embedding_dim
  embeddings = model.vq.embeddings.weight

  with torch.no_grad():
    for batch_idx, (data, _) in enumerate(val_loader):
      data = data.to(device)
      data_recon, vq_loss, _, _ = model(data)

      visual_samples = (data.cpu(), data_recon.cpu())

      data = F.interpolate(data, size=(256, 256), mode='bilinear', align_corners=False)
      data_recon = F.interpolate(data_recon, size=(256, 256), mode='bilinear', align_corners=False)

      recon_loss = F.mse_loss(data_recon, data)                                           # reconstruction loss
      percept_loss = perceptual_loss_fn(data_recon, data).mean()                          # perceptual loss

      total_recon += recon_loss.item()
      total_percept += percept_loss.item()
      total_vq += vq_loss.item()

      z = model.encoder(data)
      z_flattened = z.permute(0, 2, 3, 1).contiguous().view(-1, embedding_dim)

      distances = (torch.sum(z_flattened**2, dim=1, keepdim=True)
                    + torch.sum(embeddings**2, dim=1)
                    - 2 * torch.matmul(z_flattened, embeddings.t()))

      indices = torch.argmin(distances, dim=1)
      all_indices.append(indices.cpu())

  # percentage of used vectors
  encoding_indices = torch.cat(all_indices)
  unique_indices = torch.unique(encoding_indices)
  util_percen = len(unique_indices) / num_embeddings * 100

  # perplexity
  counts = torch.bincount(encoding_indices, minlength=num_embeddings).float()
  probs = counts / counts.sum()
  perplexity = torch.exp(-torch.sum(probs * torch.log(probs + 1e-10)))

  print(f"Reconstruction Loss: {total_recon / len(val_loader):.4f}")
  print(f"Perceptual Loss: {total_percept / len(val_loader):.4f}")
  print(f"Codebook Loss: {total_vq / len(val_loader):.4f}")

  print(f"Total Codebook Size:    {num_embeddings}")
  print(f"Unique Vectors Used:    {len(unique_indices)} / {num_embeddings}")
  print(f"Codebook Utilization:   {util_percen:.2f}%")
  print(f"Codebook Perplexity:    {perplexity.item():.2f}")

  # visual reconstruction plotting
  real_imgs, recon_imgs = visual_samples
  num_displayed_imgs = min(16, real_imgs.shape[0])

  fig, axes = plt.subplots(2, num_displayed_imgs, figsize=(num_displayed_imgs * 3, 6))

  for i in range(num_displayed_imgs):
    real_img = real_imgs[i].permute(1, 2, 0).numpy()
    recon_img = recon_imgs[i].permute(1, 2, 0).numpy()

    real_plot = ((real_img + 1) / 2).clip(0, 1)
    recon_plot = ((recon_img + 1) / 2).clip(0, 1)

    axes[0, i].imshow(real_plot)
    axes[0, i].set_title(f"original {i+1}")
    axes[0, i].axis('off')

    axes[1, i].imshow(recon_plot)
    axes[1, i].set_title(f"reconstructed {i+1}")
    axes[1, i].axis('off')

  plt.show()

In [ ]:
model.train()
save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/vq_vae'
os.makedirs(save_path, exist_ok=True)
epochs = 40

for epoch in range(epochs):
    epoch_start = time.time()
    total_loss, epoch_perplexity, avg_active_codes = 0.0, 0.0, 0.0
    total_percept, total_recon, total_vq = 0.0, 0.0, 0.0
    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device)
        optimizer.zero_grad()
        data_recon, vq_loss, perplexity, active_codes = model(data)                         # codebook loss

        data = F.interpolate(data, size=(256, 256), mode='bilinear', align_corners=False)
        data_recon = F.interpolate(data_recon, size=(256, 256), mode='bilinear', align_corners=False)

        recon_loss = F.mse_loss(data_recon, data)                                           # reconstruction loss
        percept_loss = perceptual_loss_fn(data_recon, data).mean()                          # perceptual loss
        loss = percept_loss + vq_loss + recon_loss * 0.2
        loss.backward()

        optimizer.step()

        total_loss += loss.item()
        epoch_perplexity += perplexity.item()
        avg_active_codes += active_codes

        total_percept += percept_loss
        total_recon += recon_loss
        total_vq += vq_loss
    #visual_validation(model, val_loader, device)
    print(f'====> Epoch: {epoch} | Average loss: {total_loss / len(train_loader):.4f} | Perplexity: {epoch_perplexity/len(train_loader):.2f} | Active codes: {avg_active_codes/len(train_loader):.2f}')
    print(f'----- Perceptual Loss: {total_percept / len(train_loader):.4f} | Reconstruction Loss: {total_recon / len(train_loader):.4f} | Codebook Loss: {total_vq / len(train_loader):.4f} \n')
    # save model
    #epoch_save_path = os.path.join(save_path, f"epoch_{epoch}.pth")
    #torch.save(model.state_dict(), epoch_save_path)


In [ ]:
epochs = 40
save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/vq_vae'

for epoch in range(epochs):
  epoch_save_path = os.path.join(save_path, f"epoch_{epoch}.pth")
  model = VQ_VAE().to(device)
  model.load_state_dict(torch.load(epoch_save_path, weights_only=True))
  visual_validation(model, val_loader, device)
